# Model analysis: where is the setback?

Run this after (or during) training on a checkpoint from `checkpoints/<experiment.name>/`.
Every section measures one part of the architecture on the **validation** split and the last
section turns the measurements into concrete edits for `configs/train.yaml`.
Keep `SPLIT = "val"` while tuning; use `test` only for the final report.

| Section | Question |
|---|---|
| 1 Architecture | What is trainable, how big is each part? |
| 2 Layers | Which backbone block carries product identity? Did fine-tuning help or hurt? |
| 3 Gradients | Which loss drives the shared blocks? Do losses conflict? |
| 4 I-JEPA | Do masks look right? Is the predictor learning, are targets collapsing? |
| 5 Heads | Does the discrimination head separate hard negatives better than cosine? |
| 6 Projections | What do the embeddings look like? |
| 7 Clustering | Precision/recall trade-off, merged and split products |
| 8 MLflow | Training curves |
| 9 Diagnosis | What to change in the YAML for the next run |

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import copy
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.manifold import TSNE
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader

from src.config import load_config, resolve_path
from src.data.datamodule import ProductDataModule
from src.data.dataset import ProductImageDataset
from src.data.splits import load_manifest, select_records
from src.data.transforms import eval_transform, IMAGENET_MEAN, IMAGENET_STD
from src.evaluation import analysis
from src.evaluation.metrics import clustering_metrics
from src.evaluation.plots import pair_diagnostics_figure, cluster_size_figure
from src.inference.embedding import pick_device
from src.inference.scoring import PairScorer, fit_calibration, cluster_with_scorer
from src.models.backbone import ViTBackbone
from src.training.checkpoints import checkpoint_dir, resolve_checkpoint

TRAIN_CONFIG = "configs/train.yaml"
CHECKPOINT = "best"          # best | last | path to a .ckpt
SPLIT = "val"                # val for tuning; test only for the final report
MAX_SKUS, IMAGES_PER_SKU = 100, 10
BATCH_SIZE = 64

config = load_config(TRAIN_CONFIG)
device = pick_device()
ckpt_path = resolve_checkpoint(CHECKPOINT, checkpoint_dir(config))
net, ckpt_cfg, state = analysis.load_network(ckpt_path, device)
print(f"checkpoint: {ckpt_path}\nepoch: {state['epoch']}  device: {device}")

## 1. Architecture and parameters

In [ ]:
display(pd.DataFrame(net.parameter_table()))
bb = net.backbone
print(f"backbone: {ckpt_cfg['model']['backbone']['name']}, {bb.num_blocks} blocks, dim {bb.embed_dim}, "
      f"patch {bb.patch_size}, grid {bb.grid}, trainable blocks {bb.first_trainable}..{bb.num_blocks - 1}")
print("invariance head:", net.invariance_head.net)
print("discrimination embed:", net.discrimination_head.embed)
print("energy MLP:", net.discrimination_head.energy_net)
if net.ijepa is not None:
    print("I-JEPA predictor blocks:", len(net.ijepa.predictor.predictor_blocks))

In [ ]:
# Settings that differ between this checkpoint and the current train.yaml (what the next run will change).
def flatten(d, prefix=""):
    out = {}
    for k, v in d.items():
        if str(k).startswith("_"):
            continue
        if isinstance(v, dict):
            out.update(flatten(v, f"{prefix}{k}."))
        else:
            out[f"{prefix}{k}"] = v
    return out
old, new = flatten(ckpt_cfg), flatten(config)
diff = [(k, old.get(k), new.get(k)) for k in sorted(set(old) | set(new)) if old.get(k) != new.get(k)]
display(pd.DataFrame(diff, columns=["key", "checkpoint", "train.yaml now"]) if diff else "train.yaml matches the checkpoint")

## 2. Layer-wise identity information

kNN recall@1 of each block's CLS token and mean patch token on unseen products, for the
trained backbone and for the original pretrained backbone. Shaded = trainable blocks.

In [ ]:
manifest = load_manifest(ckpt_cfg["data"]["manifest"])
frame = select_records(manifest, SPLIT, MAX_SKUS, IMAGES_PER_SKU)
root = resolve_path(ckpt_cfg["data"]["root"])
dataset = ProductImageDataset.from_manifest(frame, root, eval_transform(ckpt_cfg["data"]))
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
features = analysis.collect(net, loader, device)
labels = features["labels"]
layers = analysis.layerwise_report(features, bb.first_trainable)
print(f"{len(frame)} images, {frame['sku_id'].nunique()} products from split '{SPLIT}'")

pretrained_cfg = {**ckpt_cfg["model"]["backbone"], "pretrained": True}
reference = ViTBackbone(pretrained_cfg, ckpt_cfg["data"]["image_size"]).to(device).eval()
pretrained_recall = analysis.block_recall(reference, loader, device)
layers["pretrained_recall_at_1_cls"] = pretrained_recall + [np.nan] * (len(layers) - len(pretrained_recall))
del reference
display(layers.round(4))

blocks = layers[layers.layer.str.startswith("block_")]
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
x = np.arange(len(blocks))
ax[0].plot(x, blocks.recall_at_1_cls, "o-", label="trained, CLS")
ax[0].plot(x, blocks.recall_at_1_mean_patch, "s-", label="trained, mean patch")
ax[0].plot(x, blocks.pretrained_recall_at_1_cls, "x--", label="pretrained, CLS")
ax[0].axvspan(bb.first_trainable - 0.5, len(blocks) - 0.5, alpha=0.1, color="green")
ax[0].set_xlabel("block"); ax[0].set_ylabel("kNN recall@1"); ax[0].legend()
ax[1].bar(x, blocks.mean_token_norm); ax[1].set_xlabel("block"); ax[1].set_ylabel("mean token norm")
plt.tight_layout(); plt.show()

## 3. Per-loss gradients on the trainable blocks

Each loss is back-propagated alone on one training batch. Large differences mean one objective
dominates; a negative cosine means two objectives fight over the shared blocks.

In [ ]:
train_cfg = copy.deepcopy(ckpt_cfg); train_cfg["data"]["num_workers"] = 0
datamodule = ProductDataModule(train_cfg, patch_size=bb.patch_size); datamodule.setup()
train_batch = next(iter(datamodule.train_dataloader()))
grads, grad_cosine = analysis.per_loss_gradients(net, train_batch, ckpt_cfg, device)
display(grads.round(4)); display(grad_cosine.round(3))
cols = [c for c in grads.columns if c.startswith("block_")]
grads.set_index("loss")[cols].T.plot(kind="bar", figsize=(9, 3.5), logy=True, title="gradient norm per block and loss")
plt.tight_layout(); plt.show()

## 4. I-JEPA branch: masks and prediction quality

In [ ]:
data, masks_context, masks_target = train_batch
images = data["image"].to(device)
ijepa_stats = None
if net.ijepa is not None:
    with torch.no_grad():
        parts = net.ijepa(net.backbone, images, [m.to(device) for m in masks_context], [m.to(device) for m in masks_target])
    ijepa_stats = {k: float(v) for k, v in parts.items()}
    display(pd.Series(ijepa_stats).round(4))

    gh, gw = bb.grid
    def unnormalize(t):
        return (t.permute(1, 2, 0).cpu().numpy() * IMAGENET_STD + IMAGENET_MEAN).clip(0, 1)
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
    for col in range(4):
        grid = np.zeros(gh * gw)
        grid[masks_context[0][col].numpy()] = 1
        for t, target in enumerate(masks_target):
            grid[target[col].numpy()] = 2 + t
        axes[col].imshow(unnormalize(data["image"][col]))
        axes[col].imshow(np.kron(grid.reshape(gh, gw), np.ones((bb.patch_size, bb.patch_size))), alpha=0.45, cmap="tab10", vmin=0, vmax=9)
        axes[col].set_title("blue=context, others=targets"); axes[col].axis("off")
    plt.tight_layout(); plt.show()

## 5. Invariance head vs discrimination head

Positives = another image of the same product. Hard negatives = the nearest *different* product
in `z_inv` (what the clustering actually has to reject). The discrimination head earns its place
only if energy separates positives from hard negatives better than cosine does.

In [ ]:
pairs = analysis.pair_report(features["z_inv"], features["z_dis"], labels, net.discrimination_head)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for name, colour in (("positive", "tab:green"), ("hard_negative", "tab:red"), ("random_negative", "tab:gray")):
    ax[0].hist(pairs[name]["cosine"], bins=50, alpha=0.5, density=True, label=name, color=colour)
    ax[1].hist(pairs[name]["energy"], bins=50, alpha=0.5, density=True, label=name, color=colour)
ax[0].set_title("cosine(z_inv)  (higher = same)"); ax[1].set_title("energy(z_dis)  (lower = same)")
ax[0].legend(); plt.tight_layout(); plt.show()

y = np.r_[np.ones(len(pairs["positive"]["cosine"])), np.zeros(len(pairs["hard_negative"]["cosine"]))]
auc = pd.Series({
    "cosine, positive vs hard negative": roc_auc_score(y, np.r_[pairs["positive"]["cosine"], pairs["hard_negative"]["cosine"]]),
    "-energy, positive vs hard negative": roc_auc_score(y, -np.r_[pairs["positive"]["energy"], pairs["hard_negative"]["energy"]]),
})
display(auc.round(4))

## 6. Projections (t-SNE) for 15 products

In [ ]:
keep = np.isin(labels, np.unique(labels)[:15])
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for axis, key in zip(axes, ("pooled", "z_inv", "z_dis")):
    points = TSNE(2, perplexity=min(30, keep.sum() // 4), random_state=0, init="pca").fit_transform(features[key][keep])
    axis.scatter(points[:, 0], points[:, 1], c=labels[keep], cmap="tab20", s=14)
    axis.set_title(key); axis.set_xticks([]); axis.set_yticks([])
plt.tight_layout(); plt.show()

## 7. Clustering this split

Same procedure as training-time validation and `scripts/evaluate.py`: calibrate on these labels,
cluster, score. Then look at the worst merges (one cluster, many products) and splits
(one product, many clusters).

In [ ]:
cluster_cfg = ckpt_cfg["validation"]["clustering"]
scorer = PairScorer(features["z_inv"], features["z_dis"], net.discrimination_head)
calibration, calibration_diag = fit_calibration(scorer, labels, cluster_cfg)
pred, _ = cluster_with_scorer(scorer, cluster_cfg)
display(pd.Series(clustering_metrics(labels, pred)).to_frame("value"))
print("calibration:", {k: v for k, v in calibration.items()})
print(f"pair AUC with energy {calibration_diag['pair_auc']:.4f}  cosine only {calibration_diag['pair_auc_cosine_only']:.4f}  "
      f"kNN candidate recall {calibration_diag['candidate_recall_of_positives']:.4f}")
pair_diagnostics_figure(calibration_diag["edge_features"], calibration_diag["edge_same"], calibration["features"],
                        calibration_diag["threshold_sweep"], title=f"{SPLIT} pairs"); plt.show()
cluster_size_figure(pred, labels); plt.show()

In [ ]:
paths = [root / p for p in frame["image_path"]]

def show_row(indexes, title, n=10):
    fig, axes = plt.subplots(1, n, figsize=(1.6 * n, 2.1))
    for axis in axes:
        axis.axis("off")
    for axis, i in zip(axes, indexes[:n]):
        with Image.open(paths[i]) as im:
            axis.imshow(im.convert("RGB"))
        axis.set_title(str(labels[i]), fontsize=7)
    fig.suptitle(title, fontsize=9); plt.show()

table = pd.DataFrame({"pred": pred, "true": labels})
impurity = table.groupby("pred")["true"].nunique().sort_values(ascending=False)
for cluster in impurity.index[:3]:
    if impurity[cluster] > 1:
        show_row(np.flatnonzero(pred == cluster), f"merged: cluster {cluster} holds {impurity[cluster]} products (titles = true product id)")
spread = table.groupby("true")["pred"].nunique().sort_values(ascending=False)
for product in spread.index[:3]:
    if spread[product] > 1:
        show_row(np.flatnonzero(labels == product), f"split: product {product} spread over {spread[product]} clusters")

## 8. MLflow training curves

In [ ]:
from mlflow.tracking import MlflowClient
client = MlflowClient(tracking_uri=ckpt_cfg["mlflow"]["tracking_uri"])
run_id = state.get("mlflow_run_id")
groups = {
    "losses (epoch)": ["train/loss_epoch", "train/invariance_loss_epoch", "train/discrimination_loss_epoch", "train/ijepa_loss_epoch"],
    "validation clustering": ["val/bcubed_f1", "val/bcubed_precision", "val/bcubed_recall", "val/recall_at_1"],
    "pair scoring": ["val/pair_auc", "val/pair_auc_cosine_only", "val/candidate_recall"],
    "energy (val candidate pairs)": ["val/energy_pos", "val/energy_neg"],
}
if run_id:
    fig, axes = plt.subplots(1, len(groups), figsize=(5 * len(groups), 3.6))
    for axis, (title, keys) in zip(axes, groups.items()):
        for key in keys:
            history = client.get_metric_history(run_id, key)
            if history:
                axis.plot([h.step for h in history], [h.value for h in history], label=key.split("/")[-1])
        axis.set_title(title); axis.set_xlabel("step"); axis.legend(fontsize=7)
    plt.tight_layout(); plt.show()
    print(f"Open the full run: mlflow ui --backend-store-uri {ckpt_cfg['mlflow']['tracking_uri']}  (run {run_id})")

## 9. Diagnosis: what to change for the next run

Rules over the measurements above. Edit the named keys in `configs/train.yaml` (keep the old value
as a comment), retrain, and compare `val/bcubed_f1` in MLflow.

In [ ]:
diagnosis = analysis.diagnose(layers, grads, grad_cosine, calibration_diag, pairs, ijepa_stats,
                              margin=float(ckpt_cfg["loss"]["discrimination"]["margin"]))
pd.set_option("display.max_colwidth", 120)
display(diagnosis)